# Reaction yield: complete real-data graph workflow

Install this repository in an isolated environment with `python -m pip install -e ".[dev]"`; add Jupyter/matplotlib. Python 3.12 is recommended. Windows Application Control blocked the new RDKit wheel on the executed host, so the recorded tests/study used its existing working environment, documented in `docs/quantity-preflight.md`. All cells are initially unexecuted. Data, raw ORD records, row predictions and weights stay ignored.

In [ ]:
from pathlib import Path
import os, json, subprocess, sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Open this notebook from its own repository'
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
if str(ROOT / 'src') not in sys.path: sys.path.insert(0, str(ROOT / 'src'))
RUN_FETCH = False
RUN_TRAIN = False
RUN_INFERENCE = False
print('Repository:', ROOT)
print('Python:', sys.version.split()[0])


## Acquire and prepare the pinned real ELN source
Only input structures, roles, temperature setpoint and reported pre-reaction amounts/volumes enter predictors. The desired-product yield is extracted separately as the label. Preparation proves exact ID/group/label compatibility with the original 740-row frozen cohort. Fetch is opt-in and refuses to overwrite cached sources.

In [ ]:
if RUN_FETCH:
    assert not Path('data/external/az.parquet').exists(), 'Preserve historical raw data'
    subprocess.run([sys.executable, '-m', 'reaction_gnn.cli', 'fetch'], check=True)
    subprocess.run([sys.executable, '-m', 'reaction_gnn.cli', 'prepare'], check=True)
    from reaction_gnn.quantities import prepare_quantitative
    print(prepare_quantitative('data/external/az.parquet', 'data/processed/reactions-quantity-v2.json'))
data = Path('data/processed/reactions-quantity-v2.json')
if data.exists():
    cohort = json.loads(data.read_text(encoding='utf-8'))
    print('Real retained rows:', len(cohort['records']))
    print('Excluded records:', cohort['dropped'])


## Train with scaffold-grouped validation
The quantity plan was committed before scoring. Four GNN procedures compare old atom-role pooling with unit-safe amounts and component-balanced pooling; forests provide simpler comparators. New GNN adoption requires at least 0.5 pp mean MAE improvement and at least two fold wins. The optional command uses production code and a fresh directory. It never scores the historically observed ELN test.

In [ ]:
from datetime import datetime, timezone
if RUN_TRAIN:
    output = Path('docs') / ('notebook-quantity-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
    assert not output.exists()
    subprocess.run([sys.executable, 'train.py', '--output', str(output)], check=True)


## Compare validation results
Read recorded aggregates only. If the resumed study is not yet complete, this cell says so rather than inventing results. Full plans and histories are preserved; published rows never contain source code or reaction records.

In [ ]:
selection_path = Path('docs/quantity-v2/selection.json')
summary = []
if selection_path.exists():
    selection = json.loads(selection_path.read_text(encoding='utf-8'))
    summary = [(row['config']['name'], row['mean_validation_mae']) for row in selection['gnn_trials']]
    for row in summary: print(row)
    print('Selected GNN:', selection['selected_gnn'])
    print('Selected forest:', selection['selected_forest'])
else:
    print('Study not yet completed; no quality result claimed.')


In [ ]:
import matplotlib.pyplot as plt
if summary:
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.bar([row[0] for row in summary], [row[1] for row in summary])
    ax.set(ylabel='Validation MAE (percentage points)', title='Three scaffold-disjoint development folds')
    ax.tick_params(axis='x', rotation=25)
    plt.tight_layout()


## Inspect a production graph and reload a saved model
This checks mechanics on a development reaction and prints no source reaction. Quantity version 2 explicitly pools atoms into molecules, then molecules into role summaries. Saved checkpoints carry feature/architecture metadata. Inference is optional and accesses no heldout outcomes.

In [ ]:
if RUN_INFERENCE:
    from reaction_gnn.quantity_study import partitions
    from reaction_gnn.quantity_graph import reaction_graph
    from reaction_gnn.train import load_model, predict_record
    development, _, _ = partitions(cohort)
    graph = reaction_graph(development[0])
    print('Atom feature shape:', tuple(graph.x.shape), 'condition shape:', tuple(graph.conditions.shape))
    checkpoint = Path(os.environ.get('REACTION_MODEL', 'models/quantity-v2/selected.pt'))
    assert checkpoint.exists(), 'Provide a trained ignored checkpoint first'
    prediction = predict_record(load_model(checkpoint), development[0])
    print('Predicted yield (%):', prediction)


## Evaluation boundaries
The 130 ELN test reactions were observed in earlier work, so later results are retrospective. Explicit evaluation scripts require acknowledgement and preserve existing reports. The separate 50,688-row ORD HTE source uses assay yields and narrow substrate chemistry; a hash-selected 2,048-row transfer evaluation is meaningful only after the ELN procedure is frozen and without adaptation. Neither evaluation runs implicitly in this notebook. See `Report.md` for the final aggregate comparison and measurement-domain caveats.